PDF-RAG Project

### Step1:: installing Libary

In [10]:
!pip install langchain langchain-community langchain-text-splitters \
             langchain-groq langchain-core \
             chromadb fastembed pypdf -q

In [11]:
!pip install -q \
    langchain==0.3.25 \
    langchain-core==0.3.63 \
    langchain-community==0.3.24 \
    langchain-groq==0.3.2 \
    langchain-text-splitters==0.3.8

### step 2:: Setup the API

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "*****************************************************"
print("API Key set!")

API Key set!


### Step 3: loading the Pdf

In [34]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/content/Trading in the Zone (Mark Douglas) (Z-Library).pdf"
loader = PyPDFLoader("/content/Trading in the Zone (Mark Douglas) (Z-Library).pdf")
pages = loader.load()
print(f"Loaded: {len(pages)} pages")
print(f"Page 1 Preview: {pages[0].page_content[:300]}")

Loaded: 211 pages
Page 1 Preview: 


### Step 4:: Filtering the Non Content Pages

In [35]:
import re

# Patterns that indicate a page is boilerplate or  not real content
BOILERPLATE_PATTERNS = [
    r"all rights reserved",
    r"isbn[\s\-]?\d",
    r"table of contents",
    r"www\.\w+\.(com|org|net)",
    r"sign\s+up\s+now",
    r"what.s next on your reading list",
    r"discover your next",
    r"an imprint of",
    r"printed in the united states",
    r"library of congress",
]

compiled_patterns = [re.compile(p, re.IGNORECASE) for p in BOILERPLATE_PATTERNS]
MIN_CHARS = 150

def is_non_content(page):
    text = page.page_content.strip()
    if len(text) < MIN_CHARS:
        return True
    for pattern in compiled_patterns:
        if pattern.search(text):
            return True
    return False

before = len(pages)
pages = [p for p in pages if not is_non_content(p)]
after  = len(pages)

print(f"Pages before filter : {before}")
print(f"Pages after filter  : {after}")
print(f"Dropped: {before - after} non-content pages")

Pages before filter : 211
Pages after filter  : 197
Dropped: 14 non-content pages


### Step 5:: Splits into chunk

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(pages)
print(f"Total chunks: {len(chunks)}")
print(f"Chunk 0 preview: {chunks[0].page_content[:200]}")

Total chunks: 548
Chunk 0 preview: DEDICATION
  
This book is dedicated to all of the traders I have had the pleasure of
working with over the last 18 years as a trading coach. Each of you in your
own unique way is a part of the insigh


### Step 6 Building Vector Store

In [37]:
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma.from_documents(chunks, embeddings)
print("Vector store ready!")

Vector store ready!


### Step 7:: Create Retriever

In [38]:
# MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,           # final chunks returned
        "fetch_k": 15,    # candidates fetched before MMR re-ranking
        "lambda_mult": 0.7,  # 0 = max diversity or 1 = max relevance
    },
)

# Quick test
test_results = retriever.invoke("What is this book about?")
print(f"Retrieved {len(test_results)} chunks via MMR")
for i, doc in enumerate(test_results):
    print(f"  Chunk {i+1} — page {doc.metadata.get('page','?')}: {doc.page_content[:80]}...")

Retrieved 5 chunks via MMR
  Chunk 1 — page 146: them to interpret his story in a way that negates its validity.
To take this exa...
  Chunk 2 — page 156: a creative experience. The boy is confronted with indisputable information
that ...
  Chunk 3 — page 132: What Is Objectivity?
Objectivity is a state of mind where you have conscious acc...
  Chunk 4 — page 34: a vase, on the coffee table. He is curious, which means there’s an inner
force t...
  Chunk 5 — page 144: how much of our lives is devoted to the pursuit of money.
If we agree that peopl...


### Step 8:: Build RAG Chain with Groq

In [39]:
!pip install langchain==0.3.25 langchain-groq -q

In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Tight prompt = fewer tokens used per question
prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {input}
Answer:"""
)

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)
print("RAG chain ready!")

RAG chain ready!


### Step 9:: Cache Question

In [41]:
import hashlib
import json

# In-memory cache (lives as long as this Colab session)
question_cache = {}

def cache_key(question: str) -> str:
    """Normalize + hash so 'What is RAG?' and 'what is rag?' share one entry."""
    return hashlib.sha256(question.strip().lower().encode()).hexdigest()

def ask(question: str) -> str:
    key = cache_key(question)

    #  Cache HIT then no LLM call
    if key in question_cache:
        print("[cache hit] Returning saved answer  LLM not called.")
        return question_cache[key]

    #  Cache MISS then  call LLM
    print("[cache miss] Calling LLM...")
    response = rag_chain.invoke({"input": question})
    answer   = response["answer"]

    # Show which pages were used
    pages_used = [str(d.metadata.get('page', '?')) for d in response["context"]]
    print(f"[sources] Pages: {', '.join(pages_used)}")

    # Save to cache
    question_cache[key] = answer
    return answer

print("Cache ready! Use ask('your question') to query.")
print(f"Cache size: {len(question_cache)} entries")

Cache ready! Use ask('your question') to query.
Cache size: 0 entries


## Step 10:: Asked Question

In [42]:
# First call  hits LLM
answer = ask("What is the main message of this book?")
print("\nAnswer:", answer)

[cache miss] Calling LLM...
[sources] Pages: 144, 70, 74, 97, 28

Answer: I don't know


In [43]:
# Second call with same question  cache hit, LLM not called!
answer = ask("What is the main message of this book?")
print("\nAnswer:", answer)

[cache hit] Returning saved answer  LLM not called.

Answer: I don't know


In [44]:
answer = ask("What does Mark Douglas say about fear in trading?")
print("\nAnswer:", answer)

[cache miss] Calling LLM...
[sources] Pages: 7, 91, 70, 160, 10

Answer: The underlying cause of fear in trading is the potential to define and interpret market information as threatening, which is a result of our expectations.


In [45]:
# How many answers are cached so far?
print(f"Cache has {len(question_cache)} saved answers")

Cache has 2 saved answers


In [49]:
import json

with open('/content/rag_projects.ipynb', 'r') as f:
    nb = json.load(f)

widgets = nb['metadata'].get('widgets', {})
for key, val in widgets.items():
    print(key, ' state exists:', 'state' in val)

application/vnd.jupyter.widget-state+json  state exists: True
